In [1]:

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("🚀 Libraries imported successfully!")

🚀 Libraries imported successfully!


In [4]:
# STEP 1: LOAD PRE-TRAINED TRANSFORMER MODEL + FIX

print("🔄 Loading DialoGPT-medium from Hugging Face...\n(This may take 10-30 seconds the first time)")

model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id


print("✅ Model loaded successfully and attention mask issue FIXED! 🎉")

🔄 Loading DialoGPT-medium from Hugging Face...
(This may take 10-30 seconds the first time)


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully and attention mask issue FIXED! 🎉


In [5]:
# STEP 4: CHATBOT INITIALIZATION & CONVERSATION LOOP

# Initialize conversation history
chat_history_ids = None

print("\n" + "="*70)
print("🤖 Chatbot: Hello! I am your AI assistant. How can I help you today?")
print("   Type 'exit', 'quit' or 'bye' to end the conversation.")
print("="*70)

while True:
    # Get user input
    user_input = input("\n👤 You: ").strip()

    # Exit condition
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("🤖 Chatbot: Goodbye! Have a great day! 👋")
        break

    if not user_input:
        continue

    # Prepare input for the model
    new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token,
                                          return_tensors='pt')

    # Append to chat history
    bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) \
                    if chat_history_ids is not None else new_user_input_ids


    # Create attention mask
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        no_repeat_ngram_size=3
    )


    # Decode only the new response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print(f"🤖 Chatbot: {response}")


🤖 Chatbot: Hello! I am your AI assistant. How can I help you today?
   Type 'exit', 'quit' or 'bye' to end the conversation.

👤 You: Hi
🤖 Chatbot: Hi! :D

👤 You: What is Artificial Intelligence?
🤖 Chatbot: I'm not sure, it just looks like it's a new game?

👤 You: Really?
🤖 Chatbot: Yeah, it's pretty new.

👤 You: Do you know deep learning?
🤖 Chatbot: Nope, just the basics of programming

👤 You: Very good
🤖 Chatbot: I'll look into it!

👤 You: exit
🤖 Chatbot: Goodbye! Have a great day! 👋
